In [ ]:
# stella embedding and pca
from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model
model_name = "NovaSearch/stella_en_400M_v5"
tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)    
model = AutoModel.from_pretrained(model_name,trust_remote_code=True)


In [ ]:
import torch
import numpy as np
from tqdm import tqdm

def encode_stella_batched(texts, tokenizer, model, batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu"):
    model = model.to(device)
    model.eval()
    
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            output = model(**encoded)
            # Mean pooling
            attention_mask = encoded['attention_mask'].unsqueeze(-1)
            last_hidden = output.last_hidden_state * attention_mask
            pooled = last_hidden.sum(1) / attention_mask.sum(1)
            embeddings.append(pooled.cpu())

    return torch.cat(embeddings, dim=0).numpy()


In [ ]:
import pandas as pd
df = pd.read_csv('../api_fetcher/cleaned_abstracts.csv')
df_list = df['abstracts'].tolist()

In [ ]:
df_list = df['abstracts'].tolist()

In [ ]:
embeddings = encode_stella_batched(df_list, tokenizer, model)

Encoding: 100%|██████████| 211/211 [42:23<00:00, 12.05s/it] 


In [ ]:
 # Returns a list of sentence embeddings

np.save("stella_encoded_data.npy", embeddings)

In [ ]:
# stella embedding and pca
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from tqdm import tqdm
import pandas as pd

# Load tokenizer and model
model_name = "NovaSearch/stella_en_400M_v5"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)    
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    attn_implementation="eager"  # This fixes the xformers issue
)

def encode_stella_batched(texts, tokenizer, model, batch_size=16, device=None):
    # Auto-detect device if not specified
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    
    model = model.to(device)
    model.eval()
    
    embeddings = []
    
    # Process in batches
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i:i+batch_size]
        
        # Tokenize with proper truncation
        encoded = tokenizer(
            batch, 
            padding=True, 
            truncation=True, 
            return_tensors="pt",
            max_length=512  # Explicitly set max length
        ).to(device)
        
        with torch.no_grad():
            output = model(**encoded)
            
            # Mean pooling
            attention_mask = encoded['attention_mask'].unsqueeze(-1)
            last_hidden = output.last_hidden_state * attention_mask
            pooled = last_hidden.sum(1) / attention_mask.sum(1)
            embeddings.append(pooled.cpu())
    
    return torch.cat(embeddings, dim=0).numpy()

# Load and process data
df = pd.read_csv('../api_fetcher/cleaned_abstracts.csv')
df_list = df['abstracts'].tolist()

print(f"Encoding {len(df_list)} abstracts...")
embeddings = encode_stella_batched(df_list, tokenizer, model)

# Save embeddings
np.save("stella_encoded_data.npy", embeddings)
print(f"Embeddings shape: {embeddings.shape}")
print(f"Saved to stella_encoded_data.npy")